In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import numpy as np
import matplotlib.pyplot as plt

# Load the data

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mrtontrnok/5-vehichles-for-multicategory-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the '5-vehichles-for-multicategory-classification' dataset.
Path to dataset files: /kaggle/input/5-vehichles-for-multicategory-classification


In [3]:
path = "/kaggle/input/5-vehichles-for-multicategory-classification/dataset/"

In [4]:
TRAIN_DIR = f"{path}/train"
TEST_DIR = f"{path}/test"
VAL_DIR = f"{path}/validation"

print(f"Train directory: {TRAIN_DIR}")
print(f"Test directory: {TEST_DIR}")
print(f"Validation directory: {VAL_DIR}")

Train directory: /kaggle/input/5-vehichles-for-multicategory-classification/dataset//train
Test directory: /kaggle/input/5-vehichles-for-multicategory-classification/dataset//test
Validation directory: /kaggle/input/5-vehichles-for-multicategory-classification/dataset//validation


# Data Augmentation

In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data augmentation and normalization
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)

# Prepare data generators
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(192, 192),
    batch_size=32,
    class_mode='categorical')

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(192, 192),
    batch_size=32,
    class_mode='categorical')

validation_generator = validation_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(192, 192),
    batch_size=32,
    class_mode='categorical')

print("Data generators prepared.")

Found 5418 images belonging to 5 classes.
Found 708 images belonging to 5 classes.
Found 709 images belonging to 5 classes.
Data generators prepared.


# CNN Model

In [6]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(192, 192, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5), # Add a dropout layer here with a dropout rate of 0.5
    layers.Dense(train_generator.num_classes, activation='softmax') # Use the number of classes from the generator
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='categorical_crossentropy', # Use categorical_crossentropy for one-hot encoded labels
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 190, 190, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 95, 95, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 93, 93, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 46, 46, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 44, 44, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 22, 22, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 61952)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     3,964,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,058,565 (15.48 MB)

 Trainable params: 4,058,565 (15.48 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
import math

# Define the Early Stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Define a learning rate function
def exponential_decay(epoch, learning_rate):
    initial_lrate = 0.001
    k = 0.1
    lrate = initial_lrate * math.exp(-k*epoch)
    return lrate

# Create the learning rate scheduler callback
lr_scheduler = LearningRateScheduler(exponential_decay)

# Train the model with the callback
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=60,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    callbacks=[early_stopping, lr_scheduler] # Add the callbacks here
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/60
169/169 ━━━━━━━━━━━━━━━━━━━━ 69s 374ms/step - accuracy: 0.2785 - loss: 1.6383 - val_accuracy: 0.5270 - val_loss: 1.2564 - learning_rate: 0.0010
Epoch 2/60
  1/169 ━━━━━━━━━━━━━━━━━━━━ 8s 51ms/step - accuracy: 0.3438 - loss: 1.4437

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


169/169 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.3438 - loss: 1.4437 - val_accuracy: 0.5312 - val_loss: 1.2424 - learning_rate: 9.0484e-04
Epoch 3/60
169/169 ━━━━━━━━━━━━━━━━━━━━ 59s 349ms/step - accuracy: 0.4319 - loss: 1.3469 - val_accuracy: 0.5000 - val_loss: 1.2487 - learning_rate: 8.1873e-04
Epoch 4/60
169/169 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.3438 - loss: 1.4189 - val_accuracy: 0.5114 - val_loss: 1.2102 - learning_rate: 7.4082e-04
Epoch 5/60
169/169 ━━━━━━━━━━━━━━━━━━━━ 59s 349ms/step - accuracy: 0.4875 - loss: 1.2538 - val_accuracy: 0.5440 - val_loss: 1.1193 - learning_rate: 6.7032e-04
Epoch 6/60
169/169 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.4062 - loss: 1.3931 - val_accuracy: 0.5625 - val_loss: 1.0992 - learning_rate: 6.0653e-04
Epoch 7/60
169/169 ━━━━━━━━━━━━━━━━━━━━ 82s 361ms/step - accuracy: 0.4979 - loss: 1.2104 - val_accuracy: 0.5824 - val_loss: 1.0279 - learning_rate: 5.4881e-04
Epoch 8/60
169/169 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy

In [8]:
model.evaluate(test_generator)

23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 121ms/step - accuracy: 0.7380 - loss: 0.7065


[0.7035884261131287, 0.7387005686759949]

In [9]:
model.save("my_cnn_model2.keras")
print("Model saved successfully!")

Model saved successfully!


In [10]:
from tensorflow.keras.models import load_model
import os
import numpy as np
from tensorflow.keras.preprocessing import image

# Load the saved model
loaded_model = load_model("my_cnn_model2.keras")

# Define the image path(s) you want to predict on
# You can provide a single path as a string, or multiple paths as a list of strings
image_paths_to_predict = ["/kaggle/input/5-vehichles-for-multicategory-classification/dataset/validation/motorcycle/50.png"] # Replace with your absolute path(s)

# Process each image path
for image_path_to_predict in image_paths_to_predict:
    print(f"Predicting on image: {image_path_to_predict}")

    # Load and preprocess the image
    img = image.load_img(image_path_to_predict, target_size=(192, 192))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0  # Rescale the image

    # Make a prediction
    predictions = loaded_model.predict(img_array)

    # Get the class labels from the validation generator (assuming it's still available)
    # If not, you might need to define class_labels manually based on your dataset
    # class_labels = ['bus', 'car', 'motorcycle', 'train', 'truck']
    if 'validation_generator' in globals():
        class_labels = list(validation_generator.class_indices.keys())
    else:
        # Fallback if validation_generator is not available
        print("Warning: validation_generator not found. Using hardcoded class labels.")
        class_labels = ['bus', 'car', 'motorcycle', 'train', 'truck']


    # Display the results
    print("\nPrediction probabilities:")
    for i, probability in enumerate(predictions[0]):
        print(f"{class_labels[i]}: {probability:.4f} ({probability*100:.2f}%)")

    predicted_class_index = np.argmax(predictions)
    predicted_class_label = class_labels[predicted_class_index]
    print(f"\nPredicted class: {predicted_class_label}")
    print("-" * 30) # Separator for multiple predictions

Predicting on image: /kaggle/input/5-vehichles-for-multicategory-classification/dataset/validation/motorcycle/50.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 580ms/step

Prediction probabilities:
bus: 0.0001 (0.01%)
car: 0.0036 (0.36%)
motorcycle: 0.9952 (99.52%)
train: 0.0001 (0.01%)
truck: 0.0010 (0.10%)

Predicted class: motorcycle
------------------------------
